# 01 — Data Exploration

This notebook walks through the EuroSAT RGB dataset used for our land-cover classification and OOD detection pipeline.  
We will:

1. Load the project configuration and set the global seed for reproducibility
2. Discover all known-class images and inspect the class distribution
3. Visualize sample patches from each known class and each ghost class
4. Perform a stratified 70/15/15 train/val/test split and verify zero overlap
5. Compute channel-wise normalization statistics from the training set only

All heavy logic lives in `src/` modules — this notebook only orchestrates and visualizes.

## 1. Setup

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from torchvision import transforms

from src.config import Config
from src.data.dataset import discover_images, stratified_split, EuroSATDataset
from src.data.transforms import compute_norm_stats, save_norm_stats
from src.utils.seed import set_global_seed

In [ ]:
# Load configuration and set global seed
config = Config.load('../config.yaml')
set_global_seed(config.seed)

# Resolve dataset root relative to the notebook directory
dataset_root = '../' + config.dataset_root

print(f'Seed:         {config.seed}')
print(f'Dataset root: {dataset_root}')
print(f'Known classes: {config.known_classes}')
print(f'Ghost classes: {config.ghost_classes}')
print(f'Split ratios:  train={config.train_ratio}, val={config.val_ratio}, test={config.test_ratio}')

## 2. Image Discovery

`discover_images()` walks each class directory under the dataset root, collects every valid `.jpg` file,
and assigns integer labels 0 through N-1 based on the order of the class list.

In [ ]:
# Discover all known-class images
known_paths, known_labels = discover_images(dataset_root, config.known_classes)

print(f'Total known-class images: {len(known_paths)}')
print()

# Per-class counts
label_counts = Counter(known_labels)
for idx, cls_name in enumerate(config.known_classes):
    print(f'  {idx}: {cls_name:25s} — {label_counts[idx]:,} images')

## 3. Class Distribution

A bar chart gives a quick visual check that the dataset is roughly balanced across the 6 known classes.
Imbalanced classes would require special handling (e.g., weighted sampling), so this is an important sanity check.

In [ ]:
classes = config.known_classes
counts = [label_counts[i] for i in range(len(classes))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(classes, counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Class')
ax.set_ylabel('Number of Images')
ax.set_title('Known-Class Distribution')
ax.bar_label(bars, fmt='%d', padding=3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Sample Patches — Known Classes

Displaying a few sample patches per class helps us build intuition about what the model will learn.
Each row shows 4 randomly selected patches from one known class.

In [ ]:
from PIL import Image
import random

random.seed(config.seed)
n_samples = 4

fig, axes = plt.subplots(len(config.known_classes), n_samples,
                         figsize=(n_samples * 2.5, len(config.known_classes) * 2.5))

for row, cls_name in enumerate(config.known_classes):
    # Gather paths for this class
    cls_idx = row
    cls_paths = [p for p, l in zip(known_paths, known_labels) if l == cls_idx]
    samples = random.sample(cls_paths, min(n_samples, len(cls_paths)))
    for col, path in enumerate(samples):
        img = Image.open(path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls_name, fontsize=11, fontweight='bold', loc='left')

fig.suptitle('Sample Patches — Known Classes', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Ghost Class Preview

The 4 ghost classes (HerbaceousVegetation, Pasture, PermanentCrop, River) are **never** used during training.
We show them here purely for reference — they will appear later in the unlabeled pool for OOD detection.

In [ ]:
# Discover ghost-class images
ghost_paths, ghost_labels = discover_images(dataset_root, config.ghost_classes)
print(f'Total ghost-class images: {len(ghost_paths)}')

ghost_label_counts = Counter(ghost_labels)
for idx, cls_name in enumerate(config.ghost_classes):
    print(f'  {idx}: {cls_name:25s} — {ghost_label_counts[idx]:,} images')

In [ ]:
random.seed(config.seed)

fig, axes = plt.subplots(len(config.ghost_classes), n_samples,
                         figsize=(n_samples * 2.5, len(config.ghost_classes) * 2.5))

for row, cls_name in enumerate(config.ghost_classes):
    cls_paths = [p for p, l in zip(ghost_paths, ghost_labels) if l == row]
    samples = random.sample(cls_paths, min(n_samples, len(cls_paths)))
    for col, path in enumerate(samples):
        img = Image.open(path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls_name, fontsize=11, fontweight='bold', loc='left')

fig.suptitle('Sample Patches — Ghost Classes (reference only)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Stratified Split

We split the known-class images into **train (70%)**, **val (15%)**, and **test (15%)** using
stratified sampling so that each class is proportionally represented in every split.
The split is seeded for full reproducibility.

In [ ]:
train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = stratified_split(
    known_paths, known_labels,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.seed,
)

print(f'Train: {len(train_paths):,} images')
print(f'Val:   {len(val_paths):,} images')
print(f'Test:  {len(test_paths):,} images')
print(f'Total: {len(train_paths) + len(val_paths) + len(test_paths):,}')

In [ ]:
# Per-class counts in each split
train_counts = Counter(train_labels)
val_counts = Counter(val_labels)
test_counts = Counter(test_labels)

print(f'{"Class":25s} {"Train":>7s} {"Val":>7s} {"Test":>7s}')
print('-' * 50)
for idx, cls_name in enumerate(config.known_classes):
    print(f'{cls_name:25s} {train_counts[idx]:7d} {val_counts[idx]:7d} {test_counts[idx]:7d}')

In [ ]:
# Visualize split sizes as a grouped bar chart
x = np.arange(len(config.known_classes))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, [train_counts[i] for i in range(len(config.known_classes))], width, label='Train', color='#4c72b0')
ax.bar(x,         [val_counts[i]   for i in range(len(config.known_classes))], width, label='Val',   color='#dd8452')
ax.bar(x + width, [test_counts[i]  for i in range(len(config.known_classes))], width, label='Test',  color='#55a868')

ax.set_xlabel('Class')
ax.set_ylabel('Number of Images')
ax.set_title('Stratified Split — Per-Class Counts')
ax.set_xticks(x)
ax.set_xticklabels(config.known_classes, rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

### Split Overlap Verification

A critical data-integrity check: no file should appear in more than one split.
Any overlap would constitute data leakage and invalidate our evaluation.

In [ ]:
train_set = set(train_paths)
val_set = set(val_paths)
test_set = set(test_paths)

overlap_tv = train_set & val_set
overlap_tt = train_set & test_set
overlap_vt = val_set & test_set

print(f'Train ∩ Val:  {len(overlap_tv)} overlapping files')
print(f'Train ∩ Test: {len(overlap_tt)} overlapping files')
print(f'Val ∩ Test:   {len(overlap_vt)} overlapping files')

assert len(overlap_tv) == 0, 'Data leakage: train/val overlap!'
assert len(overlap_tt) == 0, 'Data leakage: train/test overlap!'
assert len(overlap_vt) == 0, 'Data leakage: val/test overlap!'

print('\n✅ Zero overlap — splits are clean.')

## 7. Normalization Statistics

We compute per-channel (R, G, B) mean and standard deviation using **only the training set**.
This prevents information leakage from validation/test data into the normalization transform.
The statistics are saved to `outputs/norm_stats.json` for reuse in later notebooks.

In [ ]:
# Build a minimal dataset with ToTensor only (pixel values in [0, 1])
train_dataset = EuroSATDataset(
    file_paths=train_paths,
    labels=train_labels,
    transform=transforms.ToTensor(),
)

print(f'Computing normalization stats over {len(train_dataset)} training images...')
mean, std = compute_norm_stats(train_dataset)

print(f'\nChannel-wise mean: R={mean[0]:.4f}, G={mean[1]:.4f}, B={mean[2]:.4f}')
print(f'Channel-wise std:  R={std[0]:.4f}, G={std[1]:.4f}, B={std[2]:.4f}')

In [ ]:
# Save to disk
norm_stats_path = '../' + config.norm_stats_path
save_norm_stats(mean, std, norm_stats_path)
print(f'Saved normalization stats to {norm_stats_path}')

## 8. Split Verification Summary

Final sanity checks:
- Train + Val + Test sizes sum to the total number of known-class images
- No file appears in multiple splits (already verified above, repeated here for completeness)

In [ ]:
total_known = len(known_paths)
total_split = len(train_paths) + len(val_paths) + len(test_paths)

print(f'Known-class images discovered: {total_known:,}')
print(f'Train:  {len(train_paths):,}')
print(f'Val:    {len(val_paths):,}')
print(f'Test:   {len(test_paths):,}')
print(f'Sum:    {total_split:,}')
print()

assert total_split == total_known, f'Split sizes ({total_split}) != total ({total_known})!'
print('✅ Split sizes sum to total.')

# Re-verify no overlap
all_split_paths = train_paths + val_paths + test_paths
assert len(set(all_split_paths)) == len(all_split_paths), 'Duplicate file across splits!'
print('✅ No file appears in multiple splits.')

print('\n--- Data exploration complete. Ready for training. ---')